<a href="https://colab.research.google.com/github/alkadevirajbanshi-ai/Machine-Learning-On-Big-Data2/blob/main/Property.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install pyspark
!pip3 install pyspark

In [ ]:
#initialize SparkSession and installed Required Libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize SparkSession
spark = SparkSession.builder \
                    .appName("LinearRegression_spark") \
                    .master("local[*]") \
                    .config("spark.executor.memory", "4g") \
                    .config("spark.driver.memory", "2g") \
                    .config("spark.executor.cores", "2") \
                    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
                    .getOrCreate()


spark

In [ ]:
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")

Spark UI available at: http://4d6c45124e64:4040


In [ ]:
spark.sparkContext.setLogLevel("INFO")

In [ ]:
import psutil
print(f"CPU Usage: {psutil.cpu_percent()}%")
print(f"Memory Usage: {psutil.virtual_memory().percent}%")

CPU Usage: 37.9%
Memory Usage: 11.3%


In [ ]:
# Mount Gdrive
from google.colab import drive
drive.mount

<function google.colab.drive.mount(mountpoint, force_remount=False, timeout_ms=120000, readonly=False)>

In [ ]:
# Load the data from a CSV file
df = spark.read.csv("/content/property.csv", header=True, inferSchema=True)

In [ ]:
# get familiar with data
df.show()

# more info
print("Total Records",df.count())
print("Total Partitions ",df.rdd.getNumPartitions())

+--------------+------------+-------------+----------+--------+------------------+
|Square_Footage|Num_Bedrooms|Num_Bathrooms|Year_Built|Lot_Size|             Price|
+--------------+------------+-------------+----------+--------+------------------+
|          1360|           2|            3|      1953|    7860| 303948.1373854071|
|          4272|           3|            3|      1997|    5292| 860386.2685075302|
|          3592|           4|            1|      1983|    9723| 734389.7538956215|
|           966|           6|            1|      1903|    4086| 226448.8070714377|
|          4926|           6|            4|      1944|    1081|1022486.2616704078|
|          3944|           6|            2|      1938|    3542| 845638.1354384426|
|          3671|           2|            1|      1963|    5105| 748779.2192281872|
|          3419|           4|            2|      1925|    5448| 743007.2614135538|
|           630|           2|            2|      2012|    3204| 135656.4528785377|
|   

In [ ]:
feature_cols_1 = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size']

# Create a VectorAssembler to combine feature columns into a single "features" vector
assembler_1 = VectorAssembler(inputCols=feature_cols_1, outputCol="features")
df_model_1 = assembler_1.transform(df)

# Initialize and train the Linear Regression model
lr_1 = LinearRegression(featuresCol="features", labelCol="Price")
model_1 = lr_1.fit(df_model_1)

# Make predictions
predictions_1 = model_1.transform(df_model_1)

# Evaluate the model
evaluator_1 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="rmse")
rmse_1 = evaluator_1.evaluate(predictions_1)
print(f"Model 1 - Root Mean Squared Error (RMSE): {rmse_1}")

evaluator_r2_1 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="r2")
r2_1 = evaluator_r2_1.evaluate(predictions_1)
print(f"Model 1 - R-squared (R2): {r2_1}")

Model 1 - Root Mean Squared Error (RMSE): 20002.575659182323
Model 1 - R-squared (R2): 0.9941081635515541


In this exercise, we aimed to predict the Price of properties using various numerical features from the dataset. Our approach involved building three distinct Linear Regression models, each with a different combination of input features, to understand the impact of feature selection on model performance.

Model 1 (All Numerical Features): For our initial model, we opted to include all available numerical features that seemed intuitively relevant to property pricing. These were Square_Footage, Num_Bedrooms, Num_Bathrooms, Year_Built, and Lot_Size. The rationale was to establish a baseline performance using a comprehensive set of predictors and to see if all these features collectively contribute positively to the prediction accuracy.

In [ ]:
feature_cols_2 = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']

# Create a VectorAssembler
assembler_2 = VectorAssembler(inputCols=feature_cols_2, outputCol="features")
df_model_2 = assembler_2.transform(df)

# Initialize and train the Linear Regression model
lr_2 = LinearRegression(featuresCol="features", labelCol="Price")
model_2 = lr_2.fit(df_model_2)

# Make predictions
predictions_2 = model_2.transform(df_model_2)

# Evaluate the model
evaluator_2 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="rmse")
rmse_2 = evaluator_2.evaluate(predictions_2)
print(f"Model 2 - Root Mean Squared Error (RMSE): {rmse_2}")

evaluator_r2_2 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="r2")
r2_2 = evaluator_r2_2.evaluate(predictions_2)
print(f"Model 2 - R-squared (R2): {r2_2}")

Model 2 - Root Mean Squared Error (RMSE): 20311.440728304155
Model 2 - R-squared (R2): 0.9939248039307992


Model 2 (Subset: Square_Footage, Num_Bedrooms, Num_Bathrooms): This model focused on common and highly influential property characteristics: the size of the living space (Square_Footage), and the number of Num_Bedrooms and Num_Bathrooms. These features are typically strong indicators of a property's value. The choice was to see if a more focused model could achieve comparable performance, suggesting that some features in Model 1 might be less impactful or redundant.

In [ ]:
feature_cols_3 = ['Square_Footage', 'Year_Built', 'Lot_Size']

# Create a VectorAssembler
assembler_3 = VectorAssembler(inputCols=feature_cols_3, outputCol="features")
df_model_3 = assembler_3.transform(df)

# Initialize and train the Linear Regression model
lr_3 = LinearRegression(featuresCol="features", labelCol="Price")
model_3 = lr_3.fit(df_model_3)

# Make predictions
predictions_3 = model_3.transform(df_model_3)

# Evaluate the model
evaluator_3 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="rmse")
rmse_3 = evaluator_3.evaluate(predictions_3)
print(f"Model 3 - Root Mean Squared Error (RMSE): {rmse_3}")

evaluator_r2_3 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="r2")
r2_3 = evaluator_r2_3.evaluate(predictions_3)
print(f"Model 3 - R-squared (R2): {r2_3}")

Model 3 - Root Mean Squared Error (RMSE): 22004.07758117731
Model 3 - R-squared (R2): 0.9928700715087512


Model 3 (Subset: Square_Footage, Year_Built, Lot_Size): This third model explored a different combination, keeping Square_Footage as a core predictor but replacing the number of rooms with Year_Built (property age) and Lot_Size (land area). Year_Built can indicate modernity or historical value, while Lot_Size is a significant factor in property valuation, especially in certain markets. This selection aimed to investigate how these specific features, distinct from the bedroom/bathroom count, affect prediction accuracy.

Observations from Model Comparisons
Upon training and evaluating the three Linear Regression models, we observed the following performance metrics (Root Mean Squared Error - RMSE, and R-squared - R2):

Model 1 (All numerical features):

RMSE: 20002.58
R-squared (R2): 0.9941
Model 2 (Square_Footage, Num_Bedrooms, Num_Bathrooms):

RMSE: 20311.44
R-squared (R2): 0.9939
Model 3 (Square_Footage, Year_Built, Lot_Size):

RMSE: 22004.08
R-squared (R2): 0.9929
From these comparisons, it is clear that Model 1, utilizing all available numerical features, exhibited the best overall predictive performance. It achieved the lowest RMSE, meaning its predictions were, on average, closest to the actual property prices. Additionally, its R-squared value of 0.9941 is the highest, indicating that approximately 99.41% of the variance in property prices can be explained by the features included in this model. Both Model 2 and Model 3, despite using fewer features, still performed remarkably well with very high R-squared values, suggesting that the selected subsets are also strong predictors.

Challenges Faced and How They Were Resolved
During the implementation, a minor challenge arose in the evaluation phase of Model 3. Specifically, a NameError occurred because of a typo in the print statement for the R-squared value, where r2_ was mistakenly used instead of r2_3.

This issue was straightforward to resolve by correcting the variable name in the print statement to r2_3, ensuring that the correct, previously computed R-squared value was referenced and displayed. This highlights the importance of careful syntax and variable naming, even in minor details.

Insights Gained from this Exercise
This exercise provided several valuable insights into building and evaluating Linear Regression models:

Feature Importance and Synergy: While individual features like Square_Footage are often strong predictors, combining a more comprehensive set of relevant features (as in Model 1) generally leads to a slightly better model performance. Even small improvements in R-squared can be significant in real-world applications.

Trade-off between Complexity and Performance: Model 2 and Model 3 demonstrated that even with fewer features, models can still achieve high predictive accuracy. This suggests that for certain applications, a simpler model with fewer input variables might be preferred if its performance is negligibly different from a more complex one, due to benefits like easier interpretability and faster computation.

Iterative Model Development: The process of building multiple models with different feature sets and comparing their performance is an effective way to understand the data better and refine the predictive power. It's an iterative process of hypothesis testing (which features are important?) and validation.

Attention to Detail: Even a small typographical error can halt execution, underscoring the need for meticulous coding practices and thorough testing of each component of the analysis pipeline.